# CLIENT_ID = "6QUDFzHEDfZT2x9KKzI1"
# CLIENT_SECRET = "qWwDnCr1Q5"

In [ ]:
import urllib.request
import json
import pandas as pd
import datetime

# 🚨 네이버 개발자 센터에서 발급받은 KEY 입력
CLIENT_ID = "6QUDFzHEDfZT2x9KKzI1"
CLIENT_SECRET = "qWwDnCr1Q5"

def get_tpo_trend(tpo_list):
    print("🔍 데이터랩 API 호출을 시작합니다 (최대 5개 그룹 비교)...")
    
    url = "https://openapi.naver.com/v1/datalab/search"
    
    # 최근 1년 데이터 조회 세팅
    end_date = datetime.date.today()
    start_date = end_date - datetime.timedelta(days=365)
    
    keyword_groups = []
    for tpo in tpo_list:
        keyword_groups.append({
            "groupName": tpo,
            "keywords": [tpo]
        })
        
    body = json.dumps({
        "startDate": start_date.strftime("%Y-%m-%d"),
        "endDate": end_date.strftime("%Y-%m-%d"),
        "timeUnit": "month",
        "keywordGroups": keyword_groups
    })

    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", CLIENT_ID)
    request.add_header("X-Naver-Client-Secret", CLIENT_SECRET)
    request.add_header("Content-Type", "application/json")
    
    try:
        response = urllib.request.urlopen(request, data=body.encode("utf-8"))
        rescode = response.getcode()
        
        if rescode == 200:
            response_body = response.read()
            data = json.loads(response_body.decode('utf-8'))
            
            results = []
            for group in data['results']:
                title = group['title']
                for item in group['data']:
                    results.append({
                        "Keyword": title,
                        "Date": item['period'],
                        "Search_Index": item['ratio']
                    })
            
            df = pd.DataFrame(results)
            return df
        else:
            print("Error Code:" + str(rescode))
            return None
            
    except Exception as e:
        print(f"❌ API 호출 에러 발생: {e}")
        return None

if __name__ == "__main__":
    # 🎯 라이프스타일(출근, 일상, 여행) vs 코어 스포츠(골프, 요가) 비교
    target_keywords = [
        "룰루레몬 레깅스", 
        "룰루레몬 슬랙스", 
        "룰루레몬 자켓", 
        "룰루레몬 셋업",
        "룰루레몬 팬츠" # 와이드 팬츠, 조거 등 일반 바지류 포괄
    ]
    
    trend_df = get_tpo_trend(target_keywords)
    
    if trend_df is not None:
        # 결과를 보기 좋게 피벗(Pivot) 테이블로 변환
        pivot_df = trend_df.pivot(index='Date', columns='Keyword', values='Search_Index').fillna(0)
        
        # CSV 저장
        # filename = "Lululemon_Strategic_TPO_Trend2.csv"
        # pivot_df.to_csv(filename, encoding="utf-8-sig")
        # print(f"✅ 수집 완료! [{filename}] 파일이 성공적으로 저장되었습니다.")
        print("\n📊 데이터 미리보기 (최근 5개월):")
        print(pivot_df.tail())

🔍 데이터랩 API 호출을 시작합니다 (최대 5개 그룹 비교)...
✅ 수집 완료! [Lululemon_Strategic_TPO_Trend2.csv] 파일이 성공적으로 저장되었습니다.

📊 데이터 미리보기 (최근 5개월):
Keyword      룰루레몬 레깅스  룰루레몬 셋업  룰루레몬 슬랙스  룰루레몬 자켓  룰루레몬 팬츠
Date                                                      
2025-12-01   63.19499  1.33058   0.69697  4.10264  3.83335
2026-01-01  100.00000  1.31474   0.99794  3.60367  3.91256
2026-02-01   58.78346  1.96420   0.92665  5.38571  4.22936
2026-03-01   84.15967  2.63741   2.10676  8.80722  5.42531
2026-04-01   39.81466  0.68113   1.59987  5.10058  3.31854


In [ ]:
# 1. 필수 라이브러리 설치 (자료 3회차 기준) [cite: 1240]
# !pip install curl_cffi pandas

from curl_cffi import requests
import pandas as pd
from datetime import datetime
import time

def lululemon_final_solution():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 교안 기반 봇 탐지 우회 모드 가동")
    
    # 2. 세션 생성 및 TLS 핑거프린트 위장 (자료 3회차) [cite: 1256, 1280]
    session = requests.Session(impersonate="chrome")
    
    # 3. 필수 헤더 설정 (자료 3회차 - 6페이지 템플릿 적용) [cite: 1191, 1197, 1202, 1210]
    # pseudo-headers(:로 시작)는 반드시 제외 [cite: 1105]
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Referer": "https://www.lululemon.co.kr/ko-kr/c/womens-bestsellers",
        "X-Requested-With": "XMLHttpRequest" # AJAX 요청임을 서버에 알림 [cite: 1177]
    }
    
    all_data = []
    # 4. 내부 API 엔드포인트 및 페이지네이션 (자료 2회차) [cite: 576, 588]
    # 1페이지당 40개이므로 3페이지(120개)를 수집하여 95개 추출
    for page_num in [1, 2, 3]:
        # 사용자가 제공한 Network 탭 분석 URL 패턴 적용
        if page_num == 1:
            url = "https://www.lululemon.co.kr/ko-kr/c/womens-bestsellers"
        else:
            url = f"https://www.lululemon.co.kr/on/demandware.store/Sites-KR-Site/ko_KR/Search-UpdateGrid?cgid=womens-collections-bestsellers&page={page_num}&updatePageSize=true"
        
        print(f"📄 {page_num}페이지 수집 시도 중...")
        
        try:
            # TLS 우회 요청 실행 [cite: 1247]
            response = session.get(url, headers=headers, timeout=30)
            
            if response.status_code == 200: # 성공 코드 확인 [cite: 147]
                # 정적 HTML 파싱을 위한 BeautifulSoup4 활용 (자료 4회차) [cite: 1641]
                from bs4 import BeautifulSoup
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # CSS 선택자로 요소 식별 (자료 4회차) [cite: 1783]
                products = soup.select('.product-tile')
                print(f"   -> {len(products)}개 제품 발견")
                
                for product in products:
                    name_elem = product.select_one('.pdp-link')
                    price_elem = product.select_one('.price')
                    
                    if name_elem:
                        # 텍스트 추출 및 정제 (자료 4회차) [cite: 1668, 1853]
                        all_data.append({
                            "제품명": name_elem.get_text().strip(),
                            "가격": price_elem.get_text().strip().replace('\n', ' ') if price_elem else "N/A"
                        })
            else:
                print(f"   ⚠️ 응답 실패: {response.status_code}") [cite: 148, 149]
                
            # 요청 간 간격 준수 (자료 2회차 윤리 지침) [cite: 630]
            time.sleep(2) 
            
        except Exception as e:
            print(f"   ⚠️ 오류 발생: {e}") [cite: 1828]
            continue

    # 5. 데이터 구조화 및 저장 (자료 1회차/4회차) [cite: 258, 1872, 1875]
    if all_data:
        df = pd.DataFrame(all_data).drop_duplicates(subset=['제품명']).head(95)
        df.insert(0, '순위', range(1, len(df) + 1))
        
        # filename = f"lululemon_final_report.csv"
        # df.to_csv(filename, index=False, encoding='utf-8-sig')
        
        print("\n" + "="*50)
        print(f"🎯 최종 수집 성공: 총 {len(df)}개 (교안 우회 기법 적용)")
        # print(f"📂 파일명: {filename}")
        print("="*50)
        return df

# 실행
result = lululemon_final_solution()
if result is not None:
    display(result)

[21:49:52] 🚀 교안 기반 봇 탐지 우회 모드 가동
📄 1페이지 수집 시도 중...
   -> 40개 제품 발견
📄 2페이지 수집 시도 중...
   -> 40개 제품 발견
📄 3페이지 수집 시도 중...
   -> 15개 제품 발견

🎯 최종 수집 성공: 총 95개 (교안 우회 기법 적용)
📂 파일명: lululemon_final_report.csv


,순위,제품명,가격
0,1,올웨이즈 에포트리스 재킷,"139,000원 - ..."
1,2,스쿠바 미드라이즈 와이드 레그 팬츠 아시아 핏,"135,000원 ..."
2,3,스위프틀리 울 롱슬리브 셔츠,"138,000원"
3,4,"패스트 앤 프리 하이라이즈 타이츠 25"" 5 포켓","129,000원 - ..."
4,5,스쿠바 오버사이즈드 퍼넬넥 하프집,"159,000원"
...,...,...,...
90,91,홀드 타이트 크루넥 탱크탑 웨이스트 렝스,"67,000원"
91,92,데일리 멀티 포켓 토트백 20L,"93,000원"
92,93,Softstreme 풀집 후디,"139,000원 ..."
93,94,"룰루레몬 Align™ 팬츠 25""","97,000원 ..."


In [ ]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import ast
import urllib.parse

def lululemon_schema_final():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 표준 스키마 적용 및 수집 시작")
    
    session = requests.Session(impersonate="chrome")
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Referer": "https://www.lululemon.co.kr/ko-kr/c/womens-bestsellers",
        "X-Requested-With": "XMLHttpRequest"
    }
    
    all_rows = []
    # 1~3페이지를 순회하며 98개 전수 타격
    for page_num in [1, 2, 3]:
        if page_num == 1:
            url = "https://www.lululemon.co.kr/ko-kr/c/womens-bestsellers"
        else:
            url = f"https://www.lululemon.co.kr/on/demandware.store/Sites-KR-Site/ko_KR/Search-UpdateGrid?cgid=womens-collections-bestsellers&page={page_num}&updatePageSize=true"
        
        print(f"📄 {page_num}페이지 수집 중...")
        
        try:
            response = session.get(url, headers=headers, timeout=30)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                # 💡 핵심: 데이터 보따리가 들어있는 <a> 태그를 직접 타겟팅
                product_tiles = soup.find_all('a', attrs={'data-lulu-attributes': True})
                print(f"   -> {len(product_tiles)}개 노다지 발견")
                
                for tile in product_tiles:
                    try:
                        # 1. 보따리 해독 (문자열 -> 딕셔너리)
                        attr_str = tile.get('data-lulu-attributes')
                        data_dict = ast.literal_eval(attr_str)
                        prod_info = data_dict.get('product', {})
                        
                        # 2. 표준 스키마 매핑
                        all_rows.append({
                            "collect_date": datetime.now().strftime('%Y-%m-%d'),
                            "brand": "룰루레몬",
                            "product_id": prod_info.get('productID', 'N/A'),
                            "product_name": urllib.parse.unquote(prod_info.get('name', 'N/A')),
                            "category": prod_info.get('categoryUnifiedID', 'N/A'),
                            "price": prod_info.get('price', 0),
                            "style_id": prod_info.get('styleID', 'N/A')
                        })
                    except:
                        continue
            
            time.sleep(2) # 윤리적 수집을 위한 간격
            
        except Exception as e:
            print(f"   ⚠️ 오류: {e}")
            continue

    # 3. 데이터 정제 및 최종 출력
    if all_rows:
        df = pd.DataFrame(all_rows)
        # 중복 제거 (product_id 기준)
        df = df.drop_duplicates(subset=['product_id']).reset_index(drop=True)
        
        # 순위 부여 (베스트셀러 순서 유지)
        df.insert(0, 'rank', range(1, len(df) + 1))
        
        # filename = "lululemon_bestseller_master.csv"
        # df.to_csv(filename, index=False, encoding='utf-8-sig')
        
        print("\n" + "="*50)
        print(f"🎯 최종 수집 완료: 총 {len(df)}개")
        # print(f"📂 스키마 정렬 완료: {filename}")
        print("="*50)
        return df

# 실행
master_df = lululemon_schema_final()
if master_df is not None:
    display(master_df.head(10))

[21:53:16] 🚀 표준 스키마 적용 및 수집 시작
📄 1페이지 수집 중...
   -> 617개 노다지 발견
📄 2페이지 수집 중...
   -> 288개 노다지 발견
📄 3페이지 수집 중...
   -> 79개 노다지 발견

🎯 최종 수집 완료: 총 94개
📂 스키마 정렬 완료: lululemon_bestseller_master.csv


,rank,collect_date,brand,product_id,product_name,category,price,style_id
0,1,2026-04-15,룰루레몬,N/A,N/A,N/A,0,N/A
1,2,2026-04-15,룰루레몬,prod9370052,올웨이즈 에포트리스 재킷,womens-jackets-and-outerwear,139000,prod9370052
2,3,2026-04-15,룰루레몬,LW5FYWA,스쿠바 미드라이즈 와이드 레그 팬츠 *아시아 핏,womens-sweatpants,135000,LW5FYWA
3,4,2026-04-15,룰루레몬,prod20004127,스위프틀리 울 롱슬리브 셔츠,womens-long-sleeve-tops,138000,prod20004127
4,5,2026-04-15,룰루레몬,prod11380247,"패스트 앤 프리 하이라이즈 타이츠 25"" *5 포켓",womens-leggings,129000,prod11380247
5,6,2026-04-15,룰루레몬,prod10850250,스쿠바 오버사이즈드 퍼넬넥 하프집,womens-hoodies-and-sweatshirts,159000,prod10850250
6,7,2026-04-15,룰루레몬,LW5HI7A,룰루레몬 Align™ 팔라초 팬츠 *아시아 핏,womens-casual-pants,129000,LW5HI7A
7,8,2026-04-15,룰루레몬,prod9820343,스위프틀리 테크 숏슬리브 셔츠 2.0 *웨이스트 렝스,womens-short-sleeve-tops,93000,prod9820343
8,9,2026-04-15,룰루레몬,prod11020158,디파인 재킷 *Nulu,womens-jackets-and-outerwear,147000,prod11020158
9,10,2026-04-15,룰루레몬,prod11870681,디파인 크롭 후드 재킷 *Nulu,womens-jackets-and-outerwear,184000,prod11870681


In [ ]:
import re
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import ast
import urllib.parse

def lululemon_final_fixed():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🔍 고도화된 수집 시작 (가격 범위/컬러 카운트)")
    
    session = requests.Session(impersonate="chrome")
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    }
    
    all_rows = []
    for page_num in [1, 2, 3]:
        url = "https://www.lululemon.co.kr/ko-kr/c/womens-bestsellers" if page_num == 1 else \
              f"https://www.lululemon.co.kr/on/demandware.store/Sites-KR-Site/ko_KR/Search-UpdateGrid?cgid=womens-collections-bestsellers&page={page_num}&updatePageSize=true"
        
        try:
            response = session.get(url, headers=headers, timeout=30)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 1. 개별 상품 타일을 먼저 잡아야 합니다.
            tiles = soup.select('div.product-tile')
            print(f"📄 {page_num}페이지: {len(tiles)}개 상품 분석 중...")
            
            for tile in tiles:
                try:
                    # [기본 데이터] 기존 방식 유지
                    link_tag = tile.find('a', attrs={'data-lulu-attributes': True})
                    if not link_tag: continue
                    
                    data_dict = ast.literal_eval(link_tag.get('data-lulu-attributes'))
                    prod_info = data_dict.get('product', {})

                    # [가격 데이터] 텍스트에서 모든 가격을 추출하여 Min/Max 계산
                    # 예: "₩93,000 - ₩138,000" 또는 "₩138,000"
                    price_container = tile.select_one('.price')
                    price_text = price_container.get_text(strip=True) if price_container else ""
                    
                    # 숫자만 추출 (₩와 , 제거)
                    raw_prices = re.findall(r'₩\s?([\d,]+)', price_text)
                    numeric_prices = [int(p.replace(',', '')) for p in raw_prices]

                    if numeric_prices:
                        min_price = min(numeric_prices)
                        max_price = max(numeric_prices)
                    else:
                        min_price = max_price = prod_info.get('price', 0)

                    # [컬러 옵션] 스와치 아이콘 개수 + "+N" 텍스트 합산
                    swatches = tile.select('.swatch-list li.swatch-item')
                    color_count = len(swatches)
                    
                    # "+ 3 more" 같은 텍스트가 있는지 확인
                    more_text = tile.select_one('.swatch-list .more-swatches-text')
                    if more_text:
                        more_num = re.search(r'\d+', more_text.get_text())
                        if more_num:
                            # 기본적으로 보이는 스와치 외에 추가분을 더함 (보통 5개 이후 숨김처리됨)
                            color_count += int(more_num.group())

                    all_rows.append({
                        "collect_date": datetime.now().strftime('%Y-%m-%d'),
                        "brand": "룰루레몬",
                        "product_id": prod_info.get('productID', 'N/A'),
                        "product_name": urllib.parse.unquote(prod_info.get('name', 'N/A')),
                        "min_price": min_price,
                        "max_price": max_price,
                        "color_count": color_count if color_count > 0 else 1,
                        "style_id": prod_info.get('styleID', 'N/A')
                    })
                except Exception:
                    continue
            
            time.sleep(1.5)
            
        except Exception as e:
            print(f"⚠️ 오류 발생: {e}")

    if all_rows:
        df = pd.DataFrame(all_rows).drop_duplicates(subset=['product_id']).reset_index(drop=True)
        df.insert(0, 'rank', range(1, len(df) + 1))
        return df

# 실행 및 확인
master_df = lululemon_final_fixed()
if master_df is not None:
    # 가격 차이가 있는 상품 상위 출력 (검증용)
    display(master_df.sort_values(by='max_price', ascending=False).head(10))



[11:41:44] 🔍 고도화된 수집 시작 (가격 범위/컬러 카운트)
📄 1페이지: 40개 상품 분석 중...
📄 2페이지: 40개 상품 분석 중...
📄 3페이지: 15개 상품 분석 중...


,rank,collect_date,brand,product_id,product_name,min_price,max_price,color_count,style_id
67,68,2026-04-16,룰루레몬,prod9280037,레인 리벨 재킷,355000,355000,1,prod9280037
46,47,2026-04-16,룰루레몬,prod9750601,골 스매셔 재킷,230000,230000,1,prod9750601
58,59,2026-04-16,룰루레몬,prod11870797,크롭 유틸리티 윈드브레이커,230000,230000,1,prod11870797
43,44,2026-04-16,룰루레몬,prod11871114,브리더블 라이트웨이트 트레이닝 재킷,198000,198000,1,prod11871114
73,74,2026-04-16,룰루레몬,prod11870825,다운 필 신치 웨이스트 재킷,195000,195000,1,prod11870825
21,22,2026-04-16,룰루레몬,prod9750607,레인 체이서 재킷,195000,195000,1,prod9750607
14,15,2026-04-16,룰루레몬,LW5HIGA,데이드리프트 하이라이즈 트라우저 *아시아 핏,184000,184000,1,LW5HIGA
47,48,2026-04-16,룰루레몬,prod10440041,스쿠바 오버사이즈드 풀집,184000,184000,1,prod10440041
26,27,2026-04-16,룰루레몬,LW5GULA,댄스 스튜디오 릴랙스 핏 미드라이즈 팬츠 *아시아 핏,184000,184000,1,LW5GULA
8,9,2026-04-16,룰루레몬,prod11870681,디파인 크롭 후드 재킷 *Nulu,184000,184000,1,prod11870681


In [3]:
master_df.to_csv("/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_bestseller_snapshot.csv", index=False, encoding='utf-8-sig')

In [ ]:
import re
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import ast
import urllib.parse

def lululemon_mens_bestseller_scraper():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 남성 베스트셀러 고도화 수집 시작")
    
    session = requests.Session(impersonate="chrome")
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,all;q=0.8",
        "Referer": "https://www.lululemon.co.kr/ko-kr/c/mens-bestsellers"
    }
    
    all_rows = []
    
    # 1~3페이지 순회 (남성용 cgid 적용)
    for page_num in [1, 2, 3]:
        if page_num == 1:
            url = "https://www.lululemon.co.kr/ko-kr/c/mens-bestsellers"
        else:
            url = f"https://www.lululemon.co.kr/on/demandware.store/Sites-KR-Site/ko_KR/Search-UpdateGrid?cgid=mens-collections-bestsellers&page={page_num}&updatePageSize=true"
        
        print(f"📄 {page_num}페이지 수집 중...")
        
        try:
            response = session.get(url, headers=headers, timeout=30)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                # 제품 전체를 감싸는 타일(Tile) 단위로 먼저 찾습니다.
                product_tiles = soup.select('div.product-tile')
                print(f"   -> {len(product_tiles)}개 제품 발견")
                
                for tile in product_tiles:
                    try:
                        # 1. 기존 보따리 데이터 (ID, 기본가격 등)
                        attr_tag = tile.find('a', attrs={'data-lulu-attributes': True})
                        if not attr_tag: continue
                        
                        attr_str = attr_tag.get('data-lulu-attributes')
                        data_dict = ast.literal_eval(attr_str)
                        prod_info = data_dict.get('product', {})
                        
                        # 2. 가격 범위(Min-Max) 추출 로직
                        price_elem = tile.select_one('.price')
                        price_text = price_elem.get_text(strip=True) if price_elem else ""
                        # ₩138,000 같은 숫자 패턴 모두 찾기
                        prices = [int(p.replace(',', '')) for p in re.findall(r'₩\s?([\d,]+)', price_text)]
                        
                        min_p = min(prices) if prices else prod_info.get('price', 0)
                        max_p = max(prices) if prices else min_p

                        # 3. 컬러 옵션 카운트 (스와치 아이콘 + 숨겨진 수)
                        swatches = tile.select('.swatch-list li.swatch-item')
                        color_cnt = len(swatches)
                        
                        # "+ 5 more" 같은 텍스트가 있는지 체크
                        more_text = tile.select_one('.more-swatches-text')
                        if more_text:
                            more_match = re.search(r'\d+', more_text.get_text())
                            if more_match:
                                color_cnt += int(more_match.group())

                        all_rows.append({
                            "collect_date": datetime.now().strftime('%Y-%m-%d'),
                            "brand": "룰루레몬",
                            "product_id": prod_info.get('productID', 'N/A'),
                            "product_name": urllib.parse.unquote(prod_info.get('name', 'N/A')),
                            "min_price": min_p,
                            "max_price": max_p,
                            "color_count": color_cnt if color_cnt > 0 else 1,
                            "style_id": prod_info.get('styleID', 'N/A')
                        })
                    except Exception:
                        continue
            
            time.sleep(2)
            
        except Exception as e:
            print(f"   ⚠️ 오류: {e}")
            continue

    if all_rows:
        df = pd.DataFrame(all_rows).drop_duplicates(subset=['product_id']).reset_index(drop=True)
        df.insert(0, 'rank', range(1, len(df) + 1))
        
        # filename = f"lululemon_mens_bestseller_{datetime.now().strftime('%Y%m%d')}.csv"
        # df.to_csv(filename, index=False, encoding='utf-8-sig')
        
        print("\n" + "="*50)
        print(f"🎯 최종 수집 완료: 총 {len(df)}개")
        print("="*50)
        return df

# 실행
master_df = lululemon_mens_bestseller_scraper()
if master_df is not None:
    # 가격 차이가 있는(할인 섞인) 제품 위주로 확인
    display(master_df[master_df['min_price'] != master_df['max_price']].head())

[11:58:52] 🚀 남성 베스트셀러 고도화 수집 시작
📄 1페이지 수집 중...
   -> 40개 제품 발견
📄 2페이지 수집 중...
   -> 22개 제품 발견
📄 3페이지 수집 중...
   -> 0개 제품 발견

🎯 최종 수집 완료: 총 59개
📂 파일 저장: lululemon_mens_bestseller_20260416.csv


,rank,collect_date,brand,product_id,product_name,min_price,max_price,color_count,style_id


In [ ]:
import re
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import ast
import urllib.parse

def lululemon_full_schema():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 [카테고리 포함] 남성 베스트셀러 수집 시작")
    
    session = requests.Session(impersonate="chrome")
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Referer": "https://www.lululemon.co.kr/ko-kr/c/mens-bestsellers"
    }
    
    all_rows = []
    
    for page_num in [1, 2, 3]:
        if page_num == 1:
            url = "https://www.lululemon.co.kr/ko-kr/c/mens-bestsellers"
        else:
            url = f"https://www.lululemon.co.kr/on/demandware.store/Sites-KR-Site/ko_KR/Search-UpdateGrid?cgid=mens-collections-bestsellers&page={page_num}&updatePageSize=true"
        
        try:
            response = session.get(url, headers=headers, timeout=30)
            if response.status_code != 200: continue
            
            soup = BeautifulSoup(response.text, 'html.parser')
            product_tiles = soup.select('div.product-tile')
            print(f"📄 {page_num}페이지 분석 중... ({len(product_tiles)}개 상품)")
            
            for tile in product_tiles:
                try:
                    # 1. 메타데이터 추출 (ID, 카테고리 등)
                    attr_tag = tile.find('a', attrs={'data-lulu-attributes': True})
                    if not attr_tag: continue
                    
                    data_dict = ast.literal_eval(attr_tag.get('data-lulu-attributes'))
                    prod_info = data_dict.get('product', {})
                    
                    # 2. 가격 범위 계산 (텍스트 파싱)
                    price_elem = tile.select_one('.price')
                    price_text = price_elem.get_text(strip=True) if price_elem else ""
                    prices = [int(p.replace(',', '')) for p in re.findall(r'₩\s?([\d,]+)', price_text)]
                    
                    min_p = min(prices) if prices else prod_info.get('price', 0)
                    max_p = max(prices) if prices else min_p

                    # 3. 컬러 카운트 (스와치 합산)
                    swatches = tile.select('.swatch-list li.swatch-item')
                    color_cnt = len(swatches)
                    more_text = tile.select_one('.more-swatches-text')
                    if more_text:
                        more_match = re.search(r'\d+', more_text.get_text())
                        if more_match: color_cnt += int(more_match.group())

                    # 4. 데이터 저장
                    all_rows.append({
                        "collect_date": datetime.now().strftime('%Y-%m-%d'),
                        "brand": "룰루레몬",
                        "rank": 0, # 임시
                        "product_id": prod_info.get('productID', 'N/A'),
                        "product_name": urllib.parse.unquote(prod_info.get('name', 'N/A')),
                        "category": prod_info.get('categoryUnifiedID', 'N/A'),
                        "min_price": min_p,
                        "max_price": max_p,
                        "color_count": color_cnt if color_cnt > 0 else 1,
                        "style_id": prod_info.get('styleID', 'N/A')
                    })
                except Exception:
                    continue
            
            time.sleep(1.5)
            
        except Exception as e:
            print(f"⚠️ 오류: {e}")

    if all_rows:
        df = pd.DataFrame(all_rows).drop_duplicates(subset=['product_id']).reset_index(drop=True)
        df['rank'] = df.index + 1
        
        # filename = f"lululemon_mens_full_{datetime.now().strftime('%Y%m%d')}.csv"
        # df.to_csv(filename, index=False, encoding='utf-8-sig')
        
        return df

# 이제 실행 함수 이름이 일치합니다!
master_df = lululemon_full_schema()
if master_df is not None:
    display(master_df.head(10))

[12:04:23] 🚀 [카테고리 포함] 남성 베스트셀러 수집 시작
📄 1페이지 분석 중... (40개 상품)
📄 2페이지 분석 중... (22개 상품)
📄 3페이지 분석 중... (0개 상품)

🎯 수집 완료! 총 59개 상품이 'lululemon_mens_full_20260416.csv'으로 저장되었습니다.


,collect_date,brand,rank,product_id,product_name,category,min_price,max_price,color_count,style_id
0,2026-04-16,룰루레몬,1,prod20000427,멘즈 데이드리프트 릴랙스 핏 플리티드 트라우저 *레귤러,mens-casual-pants,184000,184000,1,prod20000427
1,2026-04-16,룰루레몬,2,prod20004439,올웨이즈 다운 푸퍼 베스트 *테크 캔버스,mens-jackets-and-outerwear,387000,387000,1,prod20004439
2,2026-04-16,룰루레몬,3,prod11520172,스테디 스테이트 크루,mens-hoodies-and-sweatshirts,169000,169000,1,prod11520172
3,2026-04-16,룰루레몬,4,prod10990033,더 매트 5mm *천연고무 사용,yoga-mats,125000,125000,1,prod10990033
4,2026-04-16,룰루레몬,5,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,mens-short-sleeve-tops,74000,74000,1,prod11710026
5,2026-04-16,룰루레몬,6,prod11750248,제로드 인 트랙 재킷,mens-jackets-and-outerwear,184000,184000,1,prod11750248
6,2026-04-16,룰루레몬,7,prod11680050,헤비웨이트 코튼 저지 티셔츠,mens-short-sleeve-tops,65000,65000,1,prod11680050
7,2026-04-16,룰루레몬,8,prod11400112,"페이스 브레이커 라이너리스 쇼츠 7""",mens-shorts,74000,74000,1,prod11400112
8,2026-04-16,룰루레몬,9,prod11670131,페이스 브레이커 재킷,mens-jackets-and-outerwear,198000,198000,1,prod11670131
9,2026-04-16,룰루레몬,10,prod8510102,소전 재킷,mens-jackets-and-outerwear,147000,147000,1,prod8510102


In [ ]:
import pandas as pd
from curl_cffi import requests
import time
from datetime import datetime

def get_all_reviews_from_csv(csv_filename):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 [{csv_filename}] 대량 리뷰 수집 시작")
    
    # 1. CSV 파일에서 상품 ID 읽어오기
    try:
        df = pd.read_csv(csv_filename)
        product_ids = df[df['product_id'] != 'N/A']['product_id'].unique()
        print(f"📦 총 {len(product_ids)}개의 상품 ID 대기 중...")
    except Exception as e:
        print(f"❌ CSV 로드 실패: {e}")
        return

    # 2. 회원님의 완벽한 세션 & 헤더 설정
    headers = {
        "accept": "*/*",
        "accept-encoding": "gzip, deflate, br, zstd",
        "accept-language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "bv-bfd-token": "12369,main_site,ko_KR", # 👈 쿠키와 결합되면 작동하는 마법의 토큰
        "origin": "https://www.lululemon.co.kr",
        "referer": "https://www.lululemon.co.kr/",
        "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"', # 맥북이시지만 윈도우로 속여도 무방합니다
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "cross-site",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    }

    session = requests.Session(impersonate='chrome')
    session.headers.update(headers)
    
    # ⭐️ 핵심: 메인 페이지를 먼저 찔러서 정상 유저 쿠키를 발급받음
    print("🍪 초기 세션 쿠키 발급 중...")
    session.get('https://www.lululemon.co.kr/')
    time.sleep(1)

    service_url = 'https://apps.bazaarvoice.com/bfd/v1/clients/lululemonkorea/api-products/cv2/resources/data/reviews.json'
    all_reviews = []

    # 3. 상품별 순회 수집 시작
    for idx, pid in enumerate(product_ids, 1):
        print(f"\n🔍 [{idx}/{len(product_ids)}] 상품 ID: {pid} 리뷰 탐색 중...")
        offset = 0
        limit = 100 # 한 번에 100개씩

        while True:
            # 회원님의 깔끔한 파라미터 구조 적용
            params = {
                "resource": "reviews",
                "action": "REVIEWS_N_STATS",
                "filter": [
                    f"productid:eq:{pid}",
                    "contentlocale:eq:zh*,en*,ja*,ko_KR,ko_KR",
                    "isratingsonly:eq:false",
                ],
                "filter_reviews": "contentlocale:eq:zh*,en*,ja*,ko_KR,ko_KR",
                "include": "authors,products,comments",
                "filteredstats": "reviews",
                "Stats": "Reviews",
                "limit": limit,
                "offset": offset,
                "limit_comments": 3,
                "sort": "ContentLocale:ko_KR",
                "apiversion": "5.5",
                "displaycode": "12369-ko_kr",
            }

            try:
                response = session.get(service_url, params=params, timeout=15)

                if response.status_code != 200:
                    print(f"   ⚠️ 요청 실패: {response.status_code}")
                    break

                data = response.json()
                
                # 회원님이 파악하신 JSON 구조 적용
                response_data = data.get('response', {})
                total = response_data.get('TotalResults', 0)
                reviews = response_data.get('Results', [])

                if not reviews:
                    print("   📭 이 상품은 작성된 리뷰가 없습니다.")
                    break

                for r in reviews:
                    all_reviews.append({
                        "collect_date": datetime.now().strftime('%Y-%m-%d'),
                        "product_id": pid,
                        "rating": r.get('Rating'),
                        "title": r.get('Title'),
                        "review_text": r.get('ReviewText'),
                        "author": r.get('UserNickname'),
                        "date": r.get('SubmissionTime'),
                        "helpful": r.get('TotalPositiveFeedbackCount'),
                        "not_helpful": r.get('TotalNegativeFeedbackCount'),
                        "locale": r.get('ContentLocale')
                    })

                print(f"   ✅ 수집 진행률: {min(offset + limit, total)} / {total}")

                offset += limit
                if offset >= total: 
                    break

                time.sleep(0.5) 

            except Exception as e:
                print(f"   ⚠️ 수집 중 오류: {e}")
                break

    # 4. 최종 저장
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        # out_name = csv_filename.replace('.csv', '_reviews_final.csv')
        # df.to_csv(out_name, index=False, encoding='utf-8-sig')
        
        print("\n" + "="*50)
        print(f"🎯 전체 수집 완료! 총 {len(df)}개 리뷰 획득")
        print("="*50)
        return df
    else:
        print("\n📭 수집된 데이터가 없습니다.")
        return None

# ==========================================
# 🚀 실행하기
# ==========================================
target_csv_file = "/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_mens_bestseller_20260416.csv" 
result_df = get_all_reviews_from_csv(target_csv_file)

if result_df is not None:
    display(result_df.head())

[21:31:39] 🚀 [/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_mens_bestseller_20260416.csv] 대량 리뷰 수집 시작
📦 총 59개의 상품 ID 대기 중...
🍪 초기 세션 쿠키 발급 중...

🔍 [1/59] 상품 ID: prod20000427 리뷰 탐색 중...
   ✅ 수집 진행률: 15 / 15

🔍 [2/59] 상품 ID: prod20004439 리뷰 탐색 중...
   ✅ 수집 진행률: 7 / 7

🔍 [3/59] 상품 ID: prod11520172 리뷰 탐색 중...
   ✅ 수집 진행률: 10 / 10

🔍 [4/59] 상품 ID: prod10990033 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 368
   ✅ 수집 진행률: 200 / 368
   ✅ 수집 진행률: 300 / 368
   ✅ 수집 진행률: 368 / 368

🔍 [5/59] 상품 ID: prod11710026 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 248
   ✅ 수집 진행률: 200 / 248
   ✅ 수집 진행률: 248 / 248

🔍 [6/59] 상품 ID: prod11750248 리뷰 탐색 중...
   ✅ 수집 진행률: 21 / 21

🔍 [7/59] 상품 ID: prod11680050 리뷰 탐색 중...
   ✅ 수집 진행률: 50 / 50

🔍 [8/59] 상품 ID: prod11400112 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 218
   ✅ 수집 진행률: 200 / 218
   ✅ 수집 진행률: 218 / 218

🔍 [9/59] 상품 ID: prod11670131 리뷰 탐색 중...
   ✅ 수집 진행률: 63 / 63

🔍 [10/59] 상품 ID: prod8510102 리뷰 탐색 중...
   ✅ 수집 진행률: 82 / 82

🔍 [11/59] 상품 ID: prod10930178 리뷰 탐색 중...


,collect_date,product_id,rating,title,review_text,author,date,helpful,not_helpful,locale
0,2026-04-16,prod20000427,5,핏 최고!,기가 막힙니다 핏도 이쁘게 떨어지고 안입은것 같은 편안함이에요 다른 컬러도 하나 더...,상냥이,2026-03-18T09:24:22.000+00:00,0,0,ko_KR
1,2026-04-16,prod20000427,5,편합니다,크게 나왔어요 한사이즈 작게 주문 추천합니다\n색상도 좋고 착용감도 많이 좋아요\n...,치치맘,2026-03-06T03:33:16.000+00:00,0,0,ko_KR
2,2026-04-16,prod20000427,5,정말 편하고 운동할때도 가끔 입습니다.,일상복으로 정말 편하고 가벼운 러닝할때도 입어요. 통이 커서 평상시에 입기도 이뻐요...,nock,2026-02-09T06:48:27.000+00:00,0,0,ko_KR
3,2026-04-16,prod20000427,5,유면한 활용성 클래식무드,핏도 편하고 허리조절할 수 있어서 좋아요. 특히 턱이 있어서 클래식 느낌을 줍니다....,NaN,2025-12-13T03:53:18.000+00:00,4,0,ko_KR
4,2026-04-16,prod20000427,5,집에서도 바지를 못갈아 입겠어요,일본 여행가서 입어보고 핏에 반해 바로 한국에와서 구매 했습니다. 일본에서도 인기있...,회사가는 근육맨,2025-12-09T08:04:23.000+00:00,0,0,ko_KR


In [ ]:
import pandas as pd
from curl_cffi import requests
import time
from datetime import datetime

def get_all_reviews_from_csv(csv_filename):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 [{csv_filename}] 대량 리뷰 수집 시작")
    
    # 1. CSV 파일에서 상품 ID 읽어오기
    try:
        df = pd.read_csv(csv_filename)
        # N/A 값 제외, 고유 product_id만 추출
        product_ids = df[df['product_id'] != 'N/A']['product_id'].dropna().unique()
        print(f"📦 총 {len(product_ids)}개의 여성용 상품 ID 대기 중...")
    except Exception as e:
        print(f"❌ CSV 로드 실패: {e}")
        return

    # 2. 세션 & 헤더 설정 (우회 핵심 로직)
    headers = {
        "accept": "*/*",
        "accept-encoding": "gzip, deflate, br, zstd",
        "accept-language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "bv-bfd-token": "12369,main_site,ko_KR", 
        "origin": "https://www.lululemon.co.kr",
        "referer": "https://www.lululemon.co.kr/",
        "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "cross-site",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    }

    session = requests.Session(impersonate='chrome')
    session.headers.update(headers)
    
    # ⭐️ 쿠키 발급을 위한 메인 페이지 1회 찌르기
    print("🍪 초기 세션 쿠키 발급 중...")
    session.get('https://www.lululemon.co.kr/')
    time.sleep(1)

    service_url = 'https://apps.bazaarvoice.com/bfd/v1/clients/lululemonkorea/api-products/cv2/resources/data/reviews.json'
    all_reviews = []

    # 3. 상품별 순회 수집 시작
    for idx, pid in enumerate(product_ids, 1):
        print(f"\n🔍 [{idx}/{len(product_ids)}] 상품 ID: {pid} 리뷰 탐색 중...")
        offset = 0
        limit = 100 # 한 번에 100개씩

        while True:
            params = {
                "resource": "reviews",
                "action": "REVIEWS_N_STATS",
                "filter": [
                    f"productid:eq:{pid}",
                    "contentlocale:eq:zh*,en*,ja*,ko_KR,ko_KR",
                    "isratingsonly:eq:false",
                ],
                "filter_reviews": "contentlocale:eq:zh*,en*,ja*,ko_KR,ko_KR",
                "include": "authors,products,comments",
                "filteredstats": "reviews",
                "Stats": "Reviews",
                "limit": limit,
                "offset": offset,
                "limit_comments": 3,
                "sort": "ContentLocale:ko_KR",
                "apiversion": "5.5",
                "displaycode": "12369-ko_kr",
            }

            try:
                response = session.get(service_url, params=params, timeout=15)

                if response.status_code != 200:
                    print(f"   ⚠️ 요청 실패: {response.status_code}")
                    break

                data = response.json()
                
                response_data = data.get('response', {})
                total = response_data.get('TotalResults', 0)
                reviews = response_data.get('Results', [])

                if not reviews:
                    print("   📭 이 상품은 작성된 리뷰가 없습니다.")
                    break

                for r in reviews:
                    all_reviews.append({
                        "collect_date": datetime.now().strftime('%Y-%m-%d'),
                        "product_id": pid,
                        "rating": r.get('Rating'),
                        "title": r.get('Title'),
                        "review_text": r.get('ReviewText'),
                        "author": r.get('UserNickname'),
                        "date": r.get('SubmissionTime'),
                        "helpful": r.get('TotalPositiveFeedbackCount'),
                        "not_helpful": r.get('TotalNegativeFeedbackCount'),
                        "locale": r.get('ContentLocale')
                    })

                print(f"   ✅ 수집 진행률: {min(offset + limit, total)} / {total}")

                offset += limit
                if offset >= total: 
                    break

                time.sleep(0.5) 

            except Exception as e:
                print(f"   ⚠️ 수집 중 오류: {e}")
                break

    # 4. 최종 저장
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        # _reviews_final 이라는 이름으로 새롭게 저장
        # out_name = csv_filename.replace('.csv', '_reviews.csv')
        # df.to_csv(out_name, index=False, encoding='utf-8-sig')
        
        print("\n" + "="*50)
        print(f"🎯 여성 리뷰 수집 완료! 총 {len(df)}개 리뷰 획득")
        print("="*50)
        return df
    else:
        print("\n📭 수집된 데이터가 없습니다.")
        return None

# ==========================================
# 🚀 실행하기 (여성용 파일 경로 적용)
# ==========================================
target_csv_file_womens = "/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_womens_bestseller_20260416.csv" 
result_df_womens = get_all_reviews_from_csv(target_csv_file_womens)

if result_df_womens is not None:
    display(result_df_womens.head())

[21:35:01] 🚀 [/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_womens_bestseller_20260416.csv] 대량 리뷰 수집 시작
📦 총 93개의 여성용 상품 ID 대기 중...
🍪 초기 세션 쿠키 발급 중...

🔍 [1/93] 상품 ID: prod9370052 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 187
   ✅ 수집 진행률: 187 / 187

🔍 [2/93] 상품 ID: LW5FYWA 리뷰 탐색 중...
   ✅ 수집 진행률: 59 / 59

🔍 [3/93] 상품 ID: prod20004127 리뷰 탐색 중...
   ✅ 수집 진행률: 22 / 22

🔍 [4/93] 상품 ID: prod11380247 리뷰 탐색 중...
   ✅ 수집 진행률: 90 / 90

🔍 [5/93] 상품 ID: prod10850250 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 698
   ✅ 수집 진행률: 200 / 698
   ✅ 수집 진행률: 300 / 698
   ✅ 수집 진행률: 400 / 698
   ✅ 수집 진행률: 500 / 698
   ✅ 수집 진행률: 600 / 698
   ✅ 수집 진행률: 698 / 698

🔍 [6/93] 상품 ID: LW5HI7A 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 156
   ✅ 수집 진행률: 156 / 156

🔍 [7/93] 상품 ID: prod9820343 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 675
   ✅ 수집 진행률: 200 / 675
   ✅ 수집 진행률: 300 / 675
   ✅ 수집 진행률: 400 / 675
   ✅ 수집 진행률: 500 / 675
   ✅ 수집 진행률: 600 / 675
   ✅ 수집 진행률: 675 / 675

🔍 [8/93] 상품 ID: prod11020158 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 547
   ✅ 

,collect_date,product_id,rating,title,review_text,author,date,helpful,not_helpful,locale
0,2026-04-16,prod9370052,5,만능 자켓,간절기에 활용하기 너무 좋아요. 컬러도 은은하게 예쁘고요. 수납? 도 좋아서 등산 ...,zabelle,2026-04-16T06:23:49.000+00:00,0,0,ko_KR
1,2026-04-16,prod9370052,4,편해요,가볍고 포근해서 간절기 용으로 잘 입고있어요 \n근데 소매부분 안쪽 아우라가 자꾸...,NaN,2026-04-14T06:13:48.000+00:00,0,0,ko_KR
2,2026-04-16,prod9370052,4,봄철에 가벼게 입기 좋은 자켓,파란색 제품을 구매하였는데 너무 이쁘고 요즘 같은 날씨에 딱 적합하여 바람막이용으로...,Kate,2026-03-29T04:51:18.000+00:00,1,0,ko_KR
3,2026-04-16,prod9370052,5,디자인이 예뻐요,체구 작은편인데 4사이즈 여유있게 맞아서 핏이 너무 예뻐요 \n모자는 빼서 입는게 ...,manggo35,2025-12-23T22:27:42.000+00:00,0,1,ko_KR
4,2026-04-16,prod9370052,5,두달 실 사용 후기,가볍고 따뜻해서 계절넘어가는 시기에도 편하게 잘 입고 다닐 수 있어요. 저는 여름에...,NaN,2025-11-30T16:21:57.000+00:00,1,0,ko_KR


## 룰루레몬 전체 상품 스냅샷

In [ ]:
import re
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import ast
import urllib.parse

def lululemon_category_snapshot(cgid="mens-clothes", max_pages=50):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 '{cgid}' 카테고리 전체 스냅샷 수집 시작")
    
    session = requests.Session(impersonate="chrome")
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        # Referer를 해당 카테고리로 동적 설정
        "Referer": f"https://www.lululemon.co.kr/ko-kr/c/{cgid}" 
    }
    
    all_rows = []
    
    # 1페이지부터 max_pages까지 무한 루프 (데이터가 끊길 때까지)
    for page_num in range(1, max_pages + 1):
        # 💡 방금 찾으신 바로 그 백엔드 API URL 패턴 적용!
        url = f"https://www.lululemon.co.kr/on/demandware.store/Sites-KR-Site/ko_KR/Search-UpdateGrid?cgid={cgid}&page={page_num}&updatePageSize=true"
        
        try:
            response = session.get(url, headers=headers, timeout=30)
            
            # 서버에서 정상적인 응답이 아니면 루프 종료
            if response.status_code != 200: 
                print(f"⚠️ {page_num}페이지에서 수집 종료 (상태 코드: {response.status_code})")
                break
                
            soup = BeautifulSoup(response.text, 'html.parser')
            product_tiles = soup.select('div.product-tile')
            
            # 💡 [핵심] 더 이상 상품 타일이 없으면 해당 카테고리의 끝에 도달한 것임!
            if not product_tiles or len(product_tiles) == 0:
                print(f"🏁 {page_num}페이지에 상품이 없습니다. 수집을 완료합니다.")
                break
                
            print(f"📄 {page_num}페이지 수집 중... ({len(product_tiles)}개 상품 발견)")
            
            # (이하 회원님의 완벽한 파싱 로직 그대로 사용)
            for tile in product_tiles:
                try:
                    attr_tag = tile.find('a', attrs={'data-lulu-attributes': True})
                    if not attr_tag: continue
                    
                    data_dict = ast.literal_eval(attr_tag.get('data-lulu-attributes'))
                    prod_info = data_dict.get('product', {})
                    
                    price_elem = tile.select_one('.price')
                    price_text = price_elem.get_text(strip=True) if price_elem else ""
                    prices = [int(p.replace(',', '')) for p in re.findall(r'₩\s?([\d,]+)', price_text)]
                    min_p = min(prices) if prices else prod_info.get('price', 0)
                    
                    all_rows.append({
                        "collect_date": datetime.now().strftime('%Y-%m-%d'),
                        "brand": "룰루레몬",
                        "product_id": prod_info.get('productID', 'N/A'),
                        "product_name": urllib.parse.unquote(prod_info.get('name', 'N/A')),
                        "category": prod_info.get('categoryUnifiedID', 'N/A'),
                        "min_price": min_p,
                        "style_id": prod_info.get('styleID', 'N/A')
                    })
                except Exception:
                    continue
            
            # 봇 차단 방지를 위한 휴식 시간
            time.sleep(1.5) 
            
        except Exception as e:
            print(f"⚠️ 오류 발생: {e}")
            break # 치명적 에러 시 루프 탈출

    # 데이터프레임 생성 및 저장
    if all_rows:
        df = pd.DataFrame(all_rows).drop_duplicates(subset=['product_id']).reset_index(drop=True)
        # filename = f"lululemon_{cgid}_snapshot.csv"
        # df.to_csv(filename, index=False, encoding='utf-8-sig')
        
        print(f"\n🎯 '{cgid}' 전체 상품 수집 완료! 총 {len(df)}개 고유 상품 저장됨.")
        return df
    else:
        print("수집된 데이터가 없습니다.")
        return None

# 실행 예시 (남성복 전체 수집)
mens_snapshot_df = lululemon_category_snapshot(cgid="mens-clothes")

[11:26:12] 🚀 'mens-clothes' 카테고리 전체 스냅샷 수집 시작
📄 1페이지 수집 중... (40개 상품 발견)
📄 2페이지 수집 중... (40개 상품 발견)
📄 3페이지 수집 중... (40개 상품 발견)
📄 4페이지 수집 중... (40개 상품 발견)
📄 5페이지 수집 중... (40개 상품 발견)
📄 6페이지 수집 중... (40개 상품 발견)
📄 7페이지 수집 중... (40개 상품 발견)
📄 8페이지 수집 중... (40개 상품 발견)
📄 9페이지 수집 중... (27개 상품 발견)
🏁 10페이지에 상품이 없습니다. 수집을 완료합니다.

🎯 'mens-clothes' 전체 상품 수집 완료! 총 346개 고유 상품 저장됨.


In [2]:
# 1. 여성 카테고리 스냅샷 수집 실행
print("👩 여성복 전체 스냅샷 수집을 시작합니다...")
womens_snapshot_df = lululemon_category_snapshot(cgid="womens-clothes", max_pages=50)

# 2. 남녀 스냅샷 데이터 안전하게 병합 (데이터가 둘 다 정상 수집되었을 때만 실행)
if mens_snapshot_df is not None and womens_snapshot_df is not None:
    
    # 1) 병합 전, 각 데이터프레임에 성별을 명시하는 파생 변수(Label)를 추가
    mens_snapshot_df['gender_label'] = 'Men'
    womens_snapshot_df['gender_label'] = 'Women'
    
    # 2) 데이터 병합 및 중복 제거
    df_full_catalog = pd.concat([mens_snapshot_df, womens_snapshot_df], ignore_index=True)
    df_full_catalog = df_full_catalog.drop_duplicates(subset=['product_id']).reset_index(drop=True)
    
    # 3) 분석용 순수 카테고리(base_category) 추출
    df_full_catalog['base_category'] = df_full_catalog['category'].str.replace('womens-', '').str.replace('mens-', '')
    
    print(f"\n📦 전체 카탈로그 병합 완료: 총 {len(df_full_catalog)}개 고유 상품")
    
    # 4) 데이터 구조가 잘 잡혔는지 눈으로 확인 (크롤러에서 저장한 min_price 컬럼 사용)
    display(df_full_catalog[['product_name', 'gender_label', 'base_category', 'min_price']].sample(5))

else:
    print("⚠️ 남성복 또는 여성복 데이터 중 누락된 것이 있어 병합을 진행할 수 없습니다.")

👩 여성복 전체 스냅샷 수집을 시작합니다...
[11:34:21] 🚀 'womens-clothes' 카테고리 전체 스냅샷 수집 시작
📄 1페이지 수집 중... (40개 상품 발견)
📄 2페이지 수집 중... (40개 상품 발견)
📄 3페이지 수집 중... (40개 상품 발견)
📄 4페이지 수집 중... (40개 상품 발견)
📄 5페이지 수집 중... (40개 상품 발견)
📄 6페이지 수집 중... (40개 상품 발견)
📄 7페이지 수집 중... (40개 상품 발견)
📄 8페이지 수집 중... (40개 상품 발견)
📄 9페이지 수집 중... (40개 상품 발견)
📄 10페이지 수집 중... (40개 상품 발견)
📄 11페이지 수집 중... (40개 상품 발견)
📄 12페이지 수집 중... (40개 상품 발견)
📄 13페이지 수집 중... (40개 상품 발견)
📄 14페이지 수집 중... (40개 상품 발견)
📄 15페이지 수집 중... (40개 상품 발견)
📄 16페이지 수집 중... (40개 상품 발견)
📄 17페이지 수집 중... (20개 상품 발견)
🏁 18페이지에 상품이 없습니다. 수집을 완료합니다.

🎯 'womens-clothes' 전체 상품 수집 완료! 총 654개 고유 상품 저장됨.

📦 전체 카탈로그 병합 완료: 총 925개 고유 상품


,product_name,gender_label,base_category,min_price
549,언더이즈 미드라이즈 T팬티 언더웨어 *3팩,Women,underwear,47000
771,루나 뉴 이어 스위프틀리 테크 롱슬리브 셔츠 2.0 *웨이스트 렝스,Women,long-sleeve-tops,87000
171,제로드 인 탱크,Men,tank-tops,67000
103,비캄 릴랙스 핏 슬리브리스 셔츠,Men,tank-tops,93000
679,페더웨이트 900 다운 필 퀼티드 재킷,Women,jackets-and-outerwear,417000


### 데이터 확인 및 추출

In [3]:
df_full_catalog.info()

<class 'pandas.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   collect_date   925 non-null    str  
 1   brand          925 non-null    str  
 2   product_id     925 non-null    str  
 3   product_name   925 non-null    str  
 4   category       925 non-null    str  
 5   min_price      925 non-null    int64
 6   style_id       925 non-null    str  
 7   gender_label   925 non-null    str  
 8   base_category  925 non-null    str  
dtypes: int64(1), str(8)
memory usage: 180.9 KB


In [ ]:
# df_full_catalog.to_csv("/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_full_snapshot.csv", index=False, encoding='utf-8-sig')

## 룰루레몬 리뷰 데이터 추출

In [6]:
import pandas as pd
from curl_cffi import requests
import time
from datetime import datetime

def collect_all_snapshot_reviews(csv_filename):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 스냅샷 전체 상품 리뷰 대량 수집 시작")
    
    # 1. 스냅샷 CSV에서 상품 ID 읽어오기
    try:
        df_snapshot = pd.read_csv(csv_filename)
        # N/A 값 제외, 고유 product_id만 추출
        product_ids = df_snapshot[df_snapshot['product_id'] != 'N/A']['product_id'].dropna().unique()
        print(f"📦 총 {len(product_ids)}개의 고유 상품 ID가 대기 중입니다...")
    except Exception as e:
        print(f"❌ CSV 로드 실패. 파일 경로를 확인해주세요: {e}")
        return

    # 2. 강력한 세션 & 헤더 설정 (우회 핵심 로직)
    headers = {
        "accept": "*/*",
        "accept-encoding": "gzip, deflate, br, zstd",
        "accept-language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "bv-bfd-token": "12369,main_site,ko_KR", 
        "origin": "https://www.lululemon.co.kr",
        "referer": "https://www.lululemon.co.kr/",
        "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"', # impersonate="chrome"과 호환되도록 유지
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "cross-site",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    }

    session = requests.Session(impersonate='chrome')
    session.headers.update(headers)
    
    # ⭐️ 쿠키 발급을 위한 메인 페이지 1회 찌르기 (필수)
    print("🍪 초기 세션 쿠키 발급 중...")
    try:
        session.get('https://www.lululemon.co.kr/', timeout=15)
        time.sleep(1)
    except Exception as e:
        print(f"⚠️ 쿠키 발급 실패 (수집은 계속 시도합니다): {e}")

    service_url = 'https://apps.bazaarvoice.com/bfd/v1/clients/lululemonkorea/api-products/cv2/resources/data/reviews.json'
    all_reviews = []

    # 3. 상품별 순회 수집 시작
    for idx, pid in enumerate(product_ids, 1):
        print(f"\n🔍 [{idx}/{len(product_ids)}] 상품 ID: {pid} 리뷰 탐색 중...")
        offset = 0
        limit = 100 # 한 번에 100개씩 호출
        
        # 상품당 최대 리뷰 수집 제한 (필요시 조절, 현재 무제한)
        while True:
            params = {
                "resource": "reviews",
                "action": "REVIEWS_N_STATS",
                "filter": [
                    f"productid:eq:{pid}",
                    "contentlocale:eq:zh*,en*,ja*,ko_KR,ko_KR", # 한국어 포함 주요 언어 
                    "isratingsonly:eq:false",
                ],
                "filter_reviews": "contentlocale:eq:zh*,en*,ja*,ko_KR,ko_KR",
                "include": "authors,products,comments",
                "filteredstats": "reviews",
                "Stats": "Reviews",
                "limit": limit,
                "offset": offset,
                "limit_comments": 3,
                "sort": "ContentLocale:ko_KR",
                "apiversion": "5.5",
                "displaycode": "12369-ko_kr",
            }

            try:
                response = session.get(service_url, params=params, timeout=20)

                if response.status_code != 200:
                    print(f"   ⚠️ API 응답 거부 (상태 코드: {response.status_code}) - 다음 상품으로 넘어갑니다.")
                    break

                data = response.json()
                
                response_data = data.get('response', {})
                total = response_data.get('TotalResults', 0)
                reviews = response_data.get('Results', [])

                if not reviews:
                    print("   📭 이 상품은 작성된 리뷰가 없습니다.")
                    break

                # 4. 분석에 필요한 핵심 데이터 파싱
                for r in reviews:
                    all_reviews.append({
                        "collect_date": datetime.now().strftime('%Y-%m-%d'),
                        "product_id": pid,
                        "rating": r.get('Rating'),
                        "title": r.get('Title', ''),
                        "review_text": r.get('ReviewText', ''),
                        "author": r.get('UserNickname', ''),
                        "date": r.get('SubmissionTime', '').split('T')[0] if r.get('SubmissionTime') else None,
                        "helpful": r.get('TotalPositiveFeedbackCount', 0),
                        "not_helpful": r.get('TotalNegativeFeedbackCount', 0),
                        "locale": r.get('ContentLocale', 'Unknown')
                    })

                print(f"   ✅ 수집 진행률: {min(offset + limit, total)} / {total} (현재까지 누적: {len(all_reviews)}건)")

                offset += limit
                
                # 모든 리뷰를 가져왔으면 루프 탈출
                if offset >= total: 
                    break

                time.sleep(0.5) # 봇 차단 방지 (필수)

            except Exception as e:
                print(f"   ⚠️ {pid} 수집 중 통신 오류 발생: {e}")
                break

    # 5. 최종 데이터프레임 저장
    if all_reviews:
        df_reviews = pd.DataFrame(all_reviews)
        
        # 한국어 리뷰만 분석에 활용할 경우 필터링 (선택 사항)
        # df_reviews = df_reviews[df_reviews['locale'] == 'ko_KR'].reset_index(drop=True)
        
        # 저장 파일명: 오늘 날짜 기준
        out_name = f"lululemon_all_snapshot_reviews_{datetime.now().strftime('%Y%m%d')}.csv"
        df_reviews.to_csv(out_name, index=False, encoding='utf-8-sig')
        
        print("\n" + "="*50)
        print(f"🎯 전체 스냅샷 리뷰 수집 완료! 총 {len(df_reviews):,}개 획득")
        print(f"📁 파일 저장 완료: {out_name}")
        print("="*50)
        return df_reviews
    else:
        print("\n📭 수집된 리뷰 데이터가 하나도 없습니다.")
        return None

# ==========================================
# 🚀 실행 영역
# ==========================================
# 올려주신 파일명을 정확히 타겟팅합니다. (경로가 다를 경우 수정 필요)
target_snapshot_file = "/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_full_snapshot.csv" 
df_all_reviews = collect_all_snapshot_reviews(target_snapshot_file)

# 수집 결과 미리보기
if df_all_reviews is not None:
    display(df_all_reviews.head())

[11:47:58] 🚀 스냅샷 전체 상품 리뷰 대량 수집 시작
📦 총 925개의 고유 상품 ID가 대기 중입니다...
🍪 초기 세션 쿠키 발급 중...

🔍 [1/925] 상품 ID: prod11710026 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 249 (현재까지 누적: 100건)
   ✅ 수집 진행률: 200 / 249 (현재까지 누적: 200건)
   ✅ 수집 진행률: 249 / 249 (현재까지 누적: 249건)

🔍 [2/925] 상품 ID: prod11860280 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 308 (현재까지 누적: 349건)
   ✅ 수집 진행률: 200 / 308 (현재까지 누적: 449건)
   ✅ 수집 진행률: 300 / 308 (현재까지 누적: 549건)
   ✅ 수집 진행률: 308 / 308 (현재까지 누적: 557건)

🔍 [3/925] 상품 ID: prod11870322 리뷰 탐색 중...
   ✅ 수집 진행률: 14 / 14 (현재까지 누적: 571건)

🔍 [4/925] 상품 ID: prod10990033 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 371 (현재까지 누적: 671건)
   ✅ 수집 진행률: 200 / 371 (현재까지 누적: 771건)
   ✅ 수집 진행률: 300 / 371 (현재까지 누적: 871건)
   ✅ 수집 진행률: 371 / 371 (현재까지 누적: 942건)

🔍 [5/925] 상품 ID: prod11400112 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 218 (현재까지 누적: 1042건)
   ✅ 수집 진행률: 200 / 218 (현재까지 누적: 1142건)
   ✅ 수집 진행률: 218 / 218 (현재까지 누적: 1160건)

🔍 [6/925] 상품 ID: prod6020330 리뷰 탐색 중...
   ✅ 수집 진행률: 100 / 116 (현재까지 누적: 1260건)
   ✅ 수집 진행률: 116 / 116 (현재까지 누적: 1276건

,collect_date,product_id,rating,title,review_text,author,date,helpful,not_helpful,locale
0,2026-04-17,prod11710026,5,부드러운 옷,더운날 입기좋은 재질 부드러운 재질 땀냄새 나지 않는 재질 가격이 사악한 재질 리...,맨사 오케,2026-04-16,0,0,ko_KR
1,2026-04-17,prod11710026,5,같은옷 3개째,같은 색상의. 옷을 3개 구매 \n\n장점 : 오늘은 뭐입지 라는 고민 필요없음 \...,spartapig,2026-04-15,0,0,ko_KR
2,2026-04-17,prod11710026,3,3점,두께감이있어서 호불호가있고 색감이화면과는 조금상이합니다. 불만족하였는데 리뷰는 길...,Fftu,2026-04-15,0,0,ko_KR
3,2026-04-17,prod11710026,5,운동할때 입기 편해요,색도 맘에 들고 운동할때 땀 흡수도 잘 되고 무엇보다 편합니다.\n다른 색상도 구매...,Vowhdals,2026-04-12,0,0,ko_KR
4,2026-04-17,prod11710026,5,너무 좋아요,러닝을 하기위해서 샀는데 너무 좋습니다 굉장히 쾌적하고 러닝후 냄새도 안나고 기능성...,한준영,2026-04-04,0,0,ko_KR


In [7]:
import re
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import ast
import urllib.parse

def lululemon_premium_snapshot(cgid="mens-clothes", max_pages=50):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 '{cgid}' 프리미엄 스냅샷 수집 시작 (7종 추가 컬럼)")
    
    session = requests.Session(impersonate="chrome")
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Referer": f"https://www.lululemon.co.kr/ko-kr/c/{cgid}" 
    }
    
    all_rows = []
    
    for page_num in range(1, max_pages + 1):
        url = f"https://www.lululemon.co.kr/on/demandware.store/Sites-KR-Site/ko_KR/Search-UpdateGrid?cgid={cgid}&page={page_num}&updatePageSize=true"
        
        try:
            response = session.get(url, headers=headers, timeout=30)
            if response.status_code != 200: break
                
            soup = BeautifulSoup(response.text, 'html.parser')
            product_tiles = soup.select('div.product-tile')
            
            if not product_tiles: break
                
            print(f"📄 {page_num}페이지 수집 중... ({len(product_tiles)}개 상품 발견)")
            
            for tile in product_tiles:
                try:
                    attr_tag = tile.find('a', attrs={'data-lulu-attributes': True})
                    if not attr_tag: continue
                    
                    data_dict = ast.literal_eval(attr_tag.get('data-lulu-attributes'))
                    prod_info = data_dict.get('product', {})
                    
                    # 1. 💰 가격 파싱 (original_price, discount_price)
                    price_elem = tile.select_one('.price')
                    price_text = price_elem.get_text(strip=True) if price_elem else ""
                    prices = [int(p.replace(',', '')) for p in re.findall(r'₩\s?([\d,]+)', price_text)]
                    
                    if len(prices) >= 2:     # 할인이 적용되어 두 개의 가격(취소선 가격, 할인가)이 있는 경우
                        original_price = max(prices)
                        discount_price = min(prices)
                    elif len(prices) == 1:   # 할인이 없는 경우
                        original_price = prices[0]
                        discount_price = prices[0]
                    else:                    # 텍스트 파싱 실패 시 JSON 백업 사용
                        original_price = prod_info.get('price', 0)
                        discount_price = original_price

                    # 2. 🎨 컬러 스와치(Swatch) 정보 파싱
                    swatch_items = tile.select('.swatch-list li.swatch-item')
                    color_count = len(swatch_items)
                    
                    # '+3' 같은 추가 컬러 더보기 텍스트 합산
                    more_text = tile.select_one('.more-swatches-text')
                    if more_text:
                        more_match = re.search(r'\d+', more_text.get_text())
                        if more_match: color_count += int(more_match.group())

                    # 개별 컬러명(raw_colors) 텍스트 추출
                    raw_colors = []
                    for swatch in swatch_items:
                        color_name = swatch.get('aria-label', '').replace('Swatches', '').strip()
                        if not color_name: # aria-label이 없다면 img alt 태그 확인
                            img = swatch.select_one('img')
                            if img: color_name = img.get('alt', '').replace('Swatches', '').strip()
                        if color_name: raw_colors.append(color_name)

                    # 3. 🎖️ 뱃지 정보 (is_new, is_best) 파싱
                    badges_text = " ".join([b.get_text(strip=True) for b in tile.select('.product-badge, .badge, .badge-text')])
                    is_new = True if '신상품' in badges_text or 'New' in badges_text else False
                    is_best = True if '베스트' in badges_text or 'Best' in badges_text else False

                    # 4. 🖼️ 이미지 URL 추출
                    img_tag = tile.select_one('img.product-tile-image') or tile.select_one('picture img')
                    image_url = img_tag.get('src', '') if img_tag else ''
                    if not image_url and 'image' in prod_info: # 이미지 태그 못 찾으면 JSON 백업
                        image_url = prod_info.get('image', '')

                    # 최종 데이터 합산
                    all_rows.append({
                        "collect_date": datetime.now().strftime('%Y-%m-%d'),
                        "brand": "룰루레몬",
                        "product_id": prod_info.get('productID', 'N/A'),
                        "product_name": urllib.parse.unquote(prod_info.get('name', 'N/A')),
                        "category": prod_info.get('categoryUnifiedID', 'N/A'),
                        "style_id": prod_info.get('styleID', 'N/A'),
                        "original_price": original_price,
                        "discount_price": discount_price,
                        "color_count": color_count if color_count > 0 else 1,
                        "raw_colors": ", ".join(raw_colors) if raw_colors else "N/A",
                        "is_new": is_new,
                        "is_best": is_best,
                        "image_url": image_url
                    })
                except Exception as e:
                    continue
            
            time.sleep(1.0) 
            
        except Exception as e:
            print(f"⚠️ 오류 발생: {e}")
            break

    if all_rows:
        df = pd.DataFrame(all_rows).drop_duplicates(subset=['product_id']).reset_index(drop=True)
        return df
    else:
        return None

# ==========================================
# 🚀 남녀 데이터 프리미엄 수집 및 병합 실행
# ==========================================
print("👨 남성복 수집 시작...")
mens_premium_df = lululemon_premium_snapshot(cgid="mens-clothes", max_pages=50)

print("\n👩 여성복 수집 시작...")
womens_premium_df = lululemon_premium_snapshot(cgid="womens-clothes", max_pages=50)

if mens_premium_df is not None and womens_premium_df is not None:
    mens_premium_df['gender_label'] = 'Men'
    womens_premium_df['gender_label'] = 'Women'
    
    df_full_premium = pd.concat([mens_premium_df, womens_premium_df], ignore_index=True)
    df_full_premium = df_full_premium.drop_duplicates(subset=['product_id']).reset_index(drop=True)
    df_full_premium['base_category'] = df_full_premium['category'].str.replace('womens-', '').str.replace('mens-', '')
    
    out_name = f"lululemon_full_premium_snapshot_{datetime.now().strftime('%Y%m%d')}.csv"
    df_full_premium.to_csv(out_name, index=False, encoding='utf-8-sig')
    
    print(f"\n📦 프리미엄 스냅샷 병합 완료! 총 {len(df_full_premium)}개 상품 ({out_name})")
    display(df_full_premium.head(3))

👨 남성복 수집 시작...
[12:20:02] 🚀 'mens-clothes' 프리미엄 스냅샷 수집 시작 (7종 추가 컬럼)
📄 1페이지 수집 중... (40개 상품 발견)
📄 2페이지 수집 중... (40개 상품 발견)
📄 3페이지 수집 중... (40개 상품 발견)
📄 4페이지 수집 중... (40개 상품 발견)
📄 5페이지 수집 중... (40개 상품 발견)
📄 6페이지 수집 중... (40개 상품 발견)
📄 7페이지 수집 중... (40개 상품 발견)
📄 8페이지 수집 중... (40개 상품 발견)
📄 9페이지 수집 중... (27개 상품 발견)

👩 여성복 수집 시작...
[12:20:27] 🚀 'womens-clothes' 프리미엄 스냅샷 수집 시작 (7종 추가 컬럼)
📄 1페이지 수집 중... (40개 상품 발견)
📄 2페이지 수집 중... (40개 상품 발견)
📄 3페이지 수집 중... (40개 상품 발견)
📄 4페이지 수집 중... (40개 상품 발견)
📄 5페이지 수집 중... (40개 상품 발견)
📄 6페이지 수집 중... (40개 상품 발견)
📄 7페이지 수집 중... (40개 상품 발견)
📄 8페이지 수집 중... (40개 상품 발견)
📄 9페이지 수집 중... (40개 상품 발견)
📄 10페이지 수집 중... (40개 상품 발견)
📄 11페이지 수집 중... (40개 상품 발견)
📄 12페이지 수집 중... (40개 상품 발견)
📄 13페이지 수집 중... (40개 상품 발견)
📄 14페이지 수집 중... (40개 상품 발견)
📄 15페이지 수집 중... (40개 상품 발견)
📄 16페이지 수집 중... (40개 상품 발견)
📄 17페이지 수집 중... (19개 상품 발견)

📦 프리미엄 스냅샷 병합 완료! 총 924개 상품 (lululemon_full_premium_snapshot_20260417.csv)


,collect_date,brand,product_id,product_name,category,style_id,original_price,discount_price,color_count,raw_colors,is_new,is_best,image_url,gender_label,base_category
0,2026-04-17,룰루레몬,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,mens-short-sleeve-tops,prod11710026,74000,74000,1,N/A,False,False,,Men,short-sleeve-tops
1,2026-04-17,룰루레몬,prod11860280,심리스 와이드 헤드밴드,hair-accessories,prod11860280,29000,29000,1,N/A,False,False,,Men,hair-accessories
2,2026-04-17,룰루레몬,prod11870322,멘즈 ShowZero™ 클래식 핏 폴로 셔츠,mens-short-sleeve-tops,prod11870322,97000,97000,1,N/A,False,False,,Men,short-sleeve-tops


In [ ]:
import pandas as pd
from curl_cffi import requests
from bs4 import BeautifulSoup
import ast
import time
from datetime import datetime

def fetch_target_product_ids(cgid_list):
    """특정 카테고리(베스트, 신상)의 상품 ID 리스트만 빠르게 추출합니다."""
    session = requests.Session(impersonate="chrome")
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    }
    
    target_ids = set() # 중복을 방지하기 위해 Set 사용
    
    for cgid in cgid_list:
        print(f"🔍 '{cgid}' 카테고리 ID 스캔 중...")
        for page_num in range(1, 10): # 베스트/신상은 보통 10페이지 이내
            url = f"https://www.lululemon.co.kr/on/demandware.store/Sites-KR-Site/ko_KR/Search-UpdateGrid?cgid={cgid}&page={page_num}&updatePageSize=true"
            try:
                response = session.get(url, headers=headers, timeout=15)
                if response.status_code != 200: break
                
                soup = BeautifulSoup(response.text, 'html.parser')
                product_tiles = soup.select('div.product-tile')
                if not product_tiles: break
                
                for tile in product_tiles:
                    try:
                        attr_tag = tile.find('a', attrs={'data-lulu-attributes': True})
                        if attr_tag:
                            data_dict = ast.literal_eval(attr_tag.get('data-lulu-attributes'))
                            pid = data_dict.get('product', {}).get('productID')
                            if pid: target_ids.add(pid)
                    except:
                        continue
                time.sleep(0.5)
            except:
                break
    return target_ids

# ==========================================
# 🚀 1. 베스트셀러 & 신상 ID 추출 (룰루레몬 공식 cgid 사용)
# ==========================================
best_cgids = ["mens-collections-bestsellers", "womens-collections-bestsellers"]
new_cgids = ["whats-new"]

print("🌟 베스트셀러 상품 ID 목록 수집...")
best_seller_ids = fetch_target_product_ids(best_cgids)

print("\n✨ 신상품(What's New) 상품 ID 목록 수집...")
new_arrival_ids = fetch_target_product_ids(new_cgids)

print(f"\n✅ 확보된 베스트셀러: {len(best_seller_ids)}개 / 신상품: {len(new_arrival_ids)}개")

# ==========================================
# 🚀 2. 기존 스냅샷에 라벨링 (교차 검증)
# ==========================================
# 가지고 계신 기존 스냅샷 파일 로드
base_file = "/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_full_snapshot.csv" 

try:
    df_snapshot = pd.read_csv(base_file)
    
    # 회원님의 아이디어: ID가 해당 리스트에 포함되어 있으면 True, 아니면 False
    df_snapshot['is_best'] = df_snapshot['product_id'].isin(best_seller_ids)
    df_snapshot['is_new'] = df_snapshot['product_id'].isin(new_arrival_ids)
    
    # # 덮어쓰기 또는 새로운 파일로 저장
    # out_name = f"lululemon_snapshot_labeled_{datetime.now().strftime('%Y%m%d')}.csv"
    # df_snapshot.to_csv(out_name, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print(f"🎯 라벨링 완료! 신상품 및 베스트셀러 태그가 완벽하게 적용되었습니다.")
    # print(f"📁 저장 완료: {out_name}")
    print("="*50)
    
    # 베스트셀러이면서 신상품인 데이터 샘플 확인
    display(df_snapshot[(df_snapshot['is_best'] == True) | (df_snapshot['is_new'] == True)].head())

except Exception as e:
    print(f"❌ 스냅샷 파일을 불러오는 중 오류가 발생했습니다: {e}")

🌟 베스트셀러 상품 ID 목록 수집...
🔍 'mens-collections-bestsellers' 카테고리 ID 스캔 중...
🔍 'womens-collections-bestsellers' 카테고리 ID 스캔 중...

✨ 신상품(What's New) 상품 ID 목록 수집...
🔍 'whats-new' 카테고리 ID 스캔 중...

✅ 확보된 베스트셀러: 142개 / 신상품: 128개

🎯 라벨링 완료! 신상품 및 베스트셀러 태그가 완벽하게 적용되었습니다.
📁 저장 완료: lululemon_snapshot_labeled_20260417.csv


,collect_date,brand,product_id,product_name,category,min_price,style_id,gender_label,base_category,is_best,is_new
0,2026-04-17,룰루레몬,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,mens-short-sleeve-tops,74000,prod11710026,Men,short-sleeve-tops,True,True
3,2026-04-17,룰루레몬,prod10990033,더 매트 5mm *천연고무 사용,yoga-mats,125000,prod10990033,Men,yoga-mats,True,True
4,2026-04-17,룰루레몬,prod11400112,"페이스 브레이커 라이너리스 쇼츠 7""",mens-shorts,74000,prod11400112,Men,shorts,True,False
5,2026-04-17,룰루레몬,prod6020330,에볼루션 숏슬리브 폴로 셔츠,mens-short-sleeve-tops,88000,prod6020330,Men,short-sleeve-tops,True,True
6,2026-04-17,룰루레몬,prod20002103,뉴 크루 백팩 22L *업데이트,bags,138000,prod20002103,Men,bags,False,True


In [5]:
import pandas as pd
from datetime import datetime

# 1. 파일 로드
df_men = pd.read_csv('/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_mens-clothes_snapshot.csv')
df_women = pd.read_csv('/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_womens-clothes_snapshot.csv')
# 이미 라벨링(is_best, is_new)이 완료된 전체 스냅샷을 베이스로 활용합니다.
df_labeled = pd.read_csv('/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_snapshot_20260417.csv')

# 2. 성별 정보 보강 (개별 파일에서 성별 라벨 추출)
df_men['gender_label'] = 'Men'
df_women['gender_label'] = 'Women'

# 3. 데이터 통합 (Union)
# 개별 성별 파일들을 먼저 합칩니다.
df_combined = pd.concat([df_men, df_women], ignore_index=True)

# 4. 기존 라벨링 데이터와 병합 (Merge)
# 'is_best', 'is_new', 'base_category' 정보를 product_id 기준으로 가져옵니다.
label_info = df_labeled[['product_id', 'base_category', 'is_best', 'is_new']].drop_duplicates()

# 전체 데이터에 라벨 정보를 붙입니다.
df_master = pd.merge(df_combined, label_info, on='product_id', how='left')

# 5. 데이터 정제 (Cleaning)
# - 중복된 상품 ID 제거 (최신 수집본 유지)
df_master = df_master.drop_duplicates(subset=['product_id'], keep='first')

# - 라벨 정보가 없는 경우 기본값(False) 채우기
df_master['is_best'] = df_master['is_best'].fillna(False)
df_master['is_new'] = df_master['is_new'].fillna(False)

# 6. 최종 파일 저장 (Mac 환경 및 엑셀 호환을 위해 utf-8-sig 사용)
output_name = "lululemon_final_master_snapshot.csv"
df_master.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"✅ 최종 마스터 스냅샷 생성 완료: {output_name}")
print(f"📊 총 상품 수: {len(df_master)}개 (남성: {len(df_master[df_master['gender_label']=='Men'])} / 여성: {len(df_master[df_master['gender_label']=='Women'])})")
display(df_master.head())

✅ 최종 마스터 스냅샷 생성 완료: lululemon_final_master_snapshot.csv
📊 총 상품 수: 925개 (남성: 346 / 여성: 579)


,collect_date,brand,product_id,product_name,category,min_price,style_id,gender_label,base_category,is_best,is_new
0,2026-04-17,룰루레몬,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,mens-short-sleeve-tops,74000,prod11710026,Men,short-sleeve-tops,True,True
1,2026-04-17,룰루레몬,prod11860280,심리스 와이드 헤드밴드,hair-accessories,29000,prod11860280,Men,hair-accessories,False,False
2,2026-04-17,룰루레몬,prod11870322,멘즈 ShowZero™ 클래식 핏 폴로 셔츠,mens-short-sleeve-tops,97000,prod11870322,Men,short-sleeve-tops,False,False
3,2026-04-17,룰루레몬,prod10990033,더 매트 5mm *천연고무 사용,yoga-mats,125000,prod10990033,Men,yoga-mats,True,True
4,2026-04-17,룰루레몬,prod11400112,"페이스 브레이커 라이너리스 쇼츠 7""",mens-shorts,74000,prod11400112,Men,shorts,True,False
